# Transform Drivers Data

1. Read bronze `drivers` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`driverId` → `driver_id`, `dateOfbirth` → `date_of_birth`)
1. Concatenate `name.givenName` and `name.familyName` to create a new column called `driver_name` and transform the value to Title Case
1. Remove duplicate records
1. Transform values of column `nationality` to Title Case
1. Write the transformed data to silver `drivers` table

In [0]:
%run ../00-common/01_Environmnet_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"
silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)

In [0]:
drives_drop_url_df = drivers_df.drop("url")

In [0]:
drivers_renamed_col_df = drives_drop_url_df.withColumnRenamed(
    "driverId", "driver_id"
).withColumnRenamed("dateOfbirth", "date_of_birth")

In [0]:
drivers_named_df = drivers_renamed_col_df.withColumn(
    "driver_name",
    F.initcap(F.concat_ws(" ", F.col("name.givenName"), F.col("name.familyName"))),
).drop("name")

In [0]:
drivers_deduped_df = drivers_named_df.dropDuplicates()

In [0]:
drivers_nationality_title_df = drivers_deduped_df.withColumn(
    "nationality", F.initcap(F.col("nationality"))
)

In [0]:
drivers_nationality_title_df.write.format("delta").mode("overwrite").saveAsTable(
    silver_table
)

In [0]:
%sql
select
  *
from
  formula1.silver.drivers